## Step 0: Environment Setup & Baseline Initialization
### *Configuring dependencies and loading the foundational model.*
In this initial cell, required pip packages (such as nbformat, plotly, and ipywidgets) were installed to ensure full compatibility with the interactive visualizations. The system paths were safely configured, the Cora dataset was loaded into memory, and the baseline Graph Convolutional Network (steerable_model) was initialized to prepare for latent space interventions.

In [1]:
%pip install nbformat>=4.2.0
%pip install --upgrade nbformat ipykernel plotly scikit-learn ipywidgets
import os
import sys
import torch
import torch.nn.functional as F
from pathlib import Path
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv


# ==========================================
# 0. UNIVERSAL PATH SAFETY BLOCK
# ==========================================
current_dir = Path.cwd()

# Safely step up the tree until we find the root of your project (where app.py lives)
while not (current_dir / 'app.py').exists() and current_dir != current_dir.parent:
    current_dir = current_dir.parent

os.chdir(current_dir)
sys.path.append(str(current_dir))
print(f"✅ Working Directory Locked at: {os.getcwd()}")

# ==========================================
# 1. LOAD DATA (DOWNLOADER PERMANENTLY DISABLED)
# ==========================================
print("📦 Loading Cora Dataset...")

# 🛑 THE ULTIMATE BYPASS: Disable the downloader entirely
Planetoid.download = lambda self: print("   -> 🛑 Internet connection blocked. Forcing local file usage.")

# Now we safely point to PROJECT2/data/Cora
data_path = current_dir / 'data' / 'Cora'
dataset = Planetoid(root=str(data_path), name='Cora')
data = dataset[0]
from torch_geometric.transforms import RandomNodeSplit

# This forces an 80/10/10 split across the 2,708 nodes
transform = RandomNodeSplit(split='train_rest', num_val=0.1, num_test=0.1)
data = transform(data)
class_names = ['Theory', 'RL', 'Gen_Algos', 'Neural_Nets', 'Prob_Methods', 'Case_Based', 'Rule_Learning']

# ==========================================
# 2. Define the Steerable Architecture
# ==========================================
class SteerableGCN(torch.nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, 64)
        self.conv2 = GCNConv(64, num_classes)

    def forward(self, x, edge_index, steering_vector=None, alpha=0.0, target_node=None, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        
        # 💉 THE PATCH: Inject the concept vector into a specific node's 'brain'
        if steering_vector is not None and target_node is not None:
            h[target_node] = h[target_node] + (alpha * steering_vector)

        out = F.dropout(h, training=self.training)
        out = self.conv2(out, edge_index)
        logits = F.log_softmax(out, dim=1)
        
        if return_hidden:
            return logits, h 
        return logits

# ==========================================
# 3. Quick Training Loop (Building the "Brain")
# ==========================================
print("🧠 Training SteerableGCN model...")
steerable_model = SteerableGCN(dataset.num_features, dataset.num_classes)
optimizer = torch.optim.Adam(steerable_model.parameters(), lr=0.01, weight_decay=5e-4)

steerable_model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = steerable_model(data.x, data.edge_index)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

steerable_model.eval()

_, pred = steerable_model(data.x, data.edge_index).max(dim=1)
correct = int(pred[data.test_mask].eq(data.y[data.test_mask]).sum().item())
acc = correct / int(data.test_mask.sum())
print(f"✅ Model trained successfully! Test Accuracy: {acc:.2%}")

zsh:1: 4.2.0 not found
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Working Directory Locked at: /Users/emaheshwari/Project2
📦 Loading Cora Dataset...
🧠 Training SteerableGCN model...
✅ Model trained successfully! Test Accuracy: 86.35%


## Steps 1 & 2: Latent Extraction & Mean Concept Vectors
### *Isolating the 64-dimensional hidden states for the topic classes.*

A forward pass was executed under evaluation mode to capture the 64-dimensional latent representations of all 2,708 papers in the network. From this raw latent space, the mean concept vectors for each distinct topic class were computed, establishing the mathematical "center" of each topic's representation to be used as reference points.

In [2]:
# ==========================================
# STEPS 1 & 2: Compute Mean Concept Vectors
# ==========================================
import torch

steerable_model.eval()
with torch.no_grad():
    # Capture the 64D hidden states for all 2708 papers
    logits, hidden_states = steerable_model(data.x, data.edge_index, return_hidden=True)

concept_vectors = {}
print("⛏️ Computing Mean Concept Vectors (Centroids):")

# Loop through all 7 topics in the Cora dataset
for class_idx in range(dataset.num_classes):
    # Find all papers that belong to this specific topic
    class_mask = (data.y == class_idx)
    
    # Calculate the average (mean) 64D vector to find the topic's "DNA"
    concept_vectors[class_idx] = hidden_states[class_mask].mean(dim=0)
    
    print(f"   ✅ {class_names[class_idx]:<15} | Computed from {class_mask.sum().item()} papers")

⛏️ Computing Mean Concept Vectors (Centroids):
   ✅ Theory          | Computed from 351 papers
   ✅ RL              | Computed from 217 papers
   ✅ Gen_Algos       | Computed from 418 papers
   ✅ Neural_Nets     | Computed from 818 papers
   ✅ Prob_Methods    | Computed from 426 papers
   ✅ Case_Based      | Computed from 298 papers
   ✅ Rule_Learning   | Computed from 180 papers


## Step 3: Adversarial Vector Generation
### *Creating a comprehensive adversarial vector field.*

This step systematically calculated the mathematical distance between every possible source and target topic class. By iterating through all 42 possible combinations, a complete field of directional adversarial vectors was generated, creating the specific "steering" instructions needed to force the model to alter its classifications.

In [3]:
# ==========================================
# STEP 3: Create ALL Adversarial Concept Vectors
# ==========================================
all_adversarial_vectors = {}

print("🧬 Generating Adversarial Vector Field (All 42 Combinations)...")

# Loop through every possible Source and Target combination
for s_idx in range(dataset.num_classes):
    for t_idx in range(dataset.num_classes):
        # We don't want to steer a topic into itself
        if s_idx == t_idx: continue
        
        # The Mathematical Attack: Target DNA - Source DNA
        vec = concept_vectors[t_idx] - concept_vectors[s_idx]
        
        # Store the vector in our dictionary using a tuple (source, target) as the key
        all_adversarial_vectors[(s_idx, t_idx)] = vec

print(f"✅ Successfully created {len(all_adversarial_vectors)} unique adversarial directions.")

🧬 Generating Adversarial Vector Field (All 42 Combinations)...
✅ Successfully created 42 unique adversarial directions.


## Steps 4 & 5: Global Patching Audit
### *Stress-testing the model with multi-intensity interventions.*

A massive robustness audit was conducted by injecting the generated adversarial vectors back into the model's hidden layers during a forward pass. The audit tested 126 distinct scenarios by scaling the injection intensity (alpha values of 0.1, 0.3, and 0.5) to measure exactly how vulnerable the model's predictions are to targeted latent perturbations.

In [4]:
# ==========================================
# STEPS 4 & 5: Global Patching Audit
# ==========================================
import pandas as pd
audit_results = []
alphas = [0.1, 0.3, 0.5]

print("🧪 Auditing model robustness across 126 adversarial scenarios...")

for (s_idx, t_idx), adv_vec in all_adversarial_vectors.items():
    # Test on all nodes from the source class
    test_nodes = (data.y == s_idx).nonzero(as_tuple=True)[0].tolist()    
    for a in alphas:
        conf_drop = 0
        for node in test_nodes:
            with torch.no_grad():
                # Get baseline and patched probabilities
                orig_p = torch.exp(steerable_model(data.x, data.edge_index)[node])[s_idx].item()
                # Apply Patch: Hidden_State + (Alpha * Adv_Vector)
                patched_logits = steerable_model(data.x, data.edge_index, adv_vec, a, node)
                patched_p = torch.exp(patched_logits[node])[s_idx].item()
                conf_drop += (orig_p - patched_p)
        
        audit_results.append({
            'Path': f"{class_names[s_idx]}➔{class_names[t_idx]}",
            'Source': class_names[s_idx], 'Target': class_names[t_idx],
            'Alpha': a, 'Avg_Conf_Drop': conf_drop / len(test_nodes)
        })

df_audit = pd.DataFrame(audit_results)
print("✅ Audit complete! Results stored in 'df_audit'.")

🧪 Auditing model robustness across 126 adversarial scenarios...
✅ Audit complete! Results stored in 'df_audit'.


## Step 6: Concept Transferability Heatmaps
### *Deploying an interactive UI to audit adversarial success rates.*

An interactive widget dashboard was built utilizing Seaborn and Matplotlib. This user interface allows stakeholders to dynamically select an injection intensity (alpha) and instantly visualize a heatmap that maps out exactly which topic classes were most vulnerable—and which were most resilient—to the targeted concept transfer.

In [5]:
# ==========================================
# STEP 6: Interactive Concept Transferability Audit
# ==========================================
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

print("🏆 Dynamic Concept Transferability Audit")

# 1. Lock the color scale to the absolute maximum drop across all alphas
# This ensures that "Dark Red" means the same thing on every frame of the slider
max_drop = df_audit['Avg_Conf_Drop'].max()

def render_heatmap(alpha_val):
    # Filter the dataframe for the selected alpha
    pivot = df_audit[df_audit['Alpha'] == alpha_val].pivot(index='Source', columns='Target', values='Avg_Conf_Drop')
    
    plt.figure(figsize=(10, 8))
    # vmin=0 and vmax=max_drop keep the visual intensity consistent
    sns.heatmap(pivot, annot=True, cmap='Reds', fmt='.1%', vmin=0, vmax=max_drop, linewidths=.5)
    plt.title(f"Vulnerability Map: Confidence Drop at Alpha={alpha_val}", pad=20, fontsize=14)
    plt.ylabel("Source Topic (Original Label)")
    plt.xlabel("Target Topic (Adversarial Direction)")
    plt.tight_layout()
    plt.show()

# 2. Get the unique alphas tested in the audit (0.1, 0.3, 0.5)
tested_alphas = sorted(df_audit['Alpha'].unique())

# 3. Create a slider widget
alpha_slider = widgets.SelectionSlider(
    options=tested_alphas,
    value=tested_alphas[-1], # Default to the highest alpha (0.5)
    description='Steering Alpha:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# 4. Link the slider to the render function
widgets.interact(render_heatmap, alpha_val=alpha_slider)

# Automatically identify the easiest path at the highest tested alpha
best_path = df_audit[df_audit['Alpha'] == tested_alphas[-1]].loc[df_audit['Avg_Conf_Drop'].idxmax()]
print(f"🚨 Most Steerable Path (at α={tested_alphas[-1]}): {best_path['Path']} (Drop: {best_path['Avg_Conf_Drop']:.2%})")

🏆 Dynamic Concept Transferability Audit


interactive(children=(SelectionSlider(description='Steering Alpha:', index=2, layout=Layout(width='400px'), op…

🚨 Most Steerable Path (at α=0.5): Case_Based➔Neural_Nets (Drop: 11.79%)


## Step 7: Interactive Steering Visualization (t-SNE)
### *Mapping the latent space distortions dynamically in 2D.*

To visually prove the success of the latent steering, a t-SNE dimensionality reduction pipeline was integrated with Plotly. This generated a highly interactive dashboard where users can select a specific source-to-target attack and watch the targeted nodes physically migrate across the latent space from their original cluster into the adversarial cluster.

In [6]:
# ==========================================
# STEP 7: Interactive Steering Visualization
# ==========================================
import torch
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Pre-calculate Hidden States for the background t-SNE
steerable_model.eval()
with torch.no_grad():
    clean_logits, all_h_orig = steerable_model(data.x, data.edge_index, return_hidden=True)
    clean_preds = clean_logits.argmax(dim=1)
    clean_probs = torch.exp(clean_logits)

# 2. Build Interactive UI
source_dropdown = widgets.Dropdown(options=class_names, description='Source:', style={'description_width': 'initial'})
target_dropdown = widgets.Dropdown(options=class_names, description='Target:', value=class_names[3], style={'description_width': 'initial'})
node_dropdown = widgets.Dropdown(description='Select Node:', style={'description_width': 'initial'})
run_button = widgets.Button(description='🎬 Visualize 0.1 Trajectory', button_style='success', layout=widgets.Layout(width='250px'))
output_area = widgets.Output()

def update_node_dropdown(*args):
    source_idx = class_names.index(source_dropdown.value)
    # Filter for nodes that the model actually predicted correctly in the test set
    candidates = (data.y == source_idx) & (clean_preds == source_idx) & data.test_mask
    valid = candidates.nonzero(as_tuple=True)[0].tolist()
    if not valid:
        node_dropdown.options = [("None found", -1)]
        return
    # Rank nodes by confidence
    node_confidences = sorted([(n, clean_probs[n, source_idx].item()) for n in valid], key=lambda x: x[1], reverse=True)
    node_dropdown.options = [(f"Node {n} ({conf:.1%})", n) for n, conf in node_confidences]
    node_dropdown.value = node_confidences[0][0]

source_dropdown.observe(update_node_dropdown, 'value')
update_node_dropdown()

# 3. Trajectory & Precision Engine
def run_steering_experiment(b):
    with output_area:
        clear_output(wait=True)
        source_name, target_name, test_node = source_dropdown.value, target_dropdown.value, node_dropdown.value
        
        if source_name == target_name or test_node == -1:
            print("⚠️ Source and Target must be different.")
            return

        source_idx, target_idx = class_names.index(source_name), class_names.index(target_name)
        
        # Use the adversarial vector already computed in Step 3
        v_adv = all_adversarial_vectors[(source_idx, target_idx)]

        # --- A. PRECISION SEARCH (0.01 increments) ---
        # Task: Find the 50% breaking point accurate to 2nd decimal
        print(f"🔬 Scanning for 50% Decision Point (0.01 precision)...")
        breaking_point = None
        for a in np.arange(0.0, 30.01, 0.01):
            with torch.no_grad():
                logits = steerable_model(data.x, data.edge_index, v_adv, a, test_node)
                if torch.exp(logits[test_node])[target_idx] >= 0.50:
                    breaking_point = a
                    break

        # --- B. ANIMATION DATA (0.1 increments 0 to 4.0) ---
        print(f"🎬 Generating 0.1 step animation (Total 41 frames)...")
        alphas_to_track = np.arange(0.0, 4.1, 0.1)
        steered_states = []
        for a in alphas_to_track:
            with torch.no_grad():
                _, h = steerable_model(data.x, data.edge_index, v_adv, a, test_node, return_hidden=True)
                steered_states.append(h[test_node].cpu().numpy())

        # t-SNE Projection
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        combined_h = np.vstack([all_h_orig.cpu().numpy()] + steered_states)
        coords = tsne.fit_transform(combined_h)
        
        orig_coords = coords[:len(data.y)]
        traj_coords = coords[len(data.y):]

        # Build Dataframe for Plotly
        frames = []
        bg_classes = [class_names[c] for c in data.y.cpu().numpy()]
        for i, a in enumerate(alphas_to_track):
            df = pd.DataFrame({'X': orig_coords[:, 0], 'Y': orig_coords[:, 1], 'Topic': bg_classes, 'Size': 1, 'Alpha': f"{a:.1f}"})
            # Inject the moving node
            df.loc[test_node, ['X', 'Y', 'Topic', 'Size']] = [traj_coords[i, 0], traj_coords[i, 1], '🚀 STEERED', 12]
            frames.append(df)
        
        plot_df = pd.concat(frames)

        # 📊 Report results
        print("="*60)
        if breaking_point:
            print(f"💥 50% BREAKING POINT: Alpha = {breaking_point:.2f}")
            print(f"   (Detected with 0.01 precision)")
        else:
            print(f"🛡️ ROBUST NODE: Did not flip to 50% even at Alpha 30.0")
        print("="*60)

        # Render Plotly Trajectory
        fig = px.scatter(plot_df, x="X", y="Y", animation_frame="Alpha", color="Topic", size="Size", size_max=15,
                         title=f"Steering Path: {source_name} ➔ {target_name} | Node {test_node}",
                         template="plotly_dark", width=950, height=650)
        fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
        fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 150
        fig.show()

run_button.on_click(run_steering_experiment)
display(widgets.VBox([widgets.HBox([source_dropdown, target_dropdown]), widgets.HBox([node_dropdown, run_button]), output_area]))

## Step 8: Artifact Export & Save State
### *Persisting adversarial vectors and audit logs for production.*

In this final step, the dynamic project directory was safely resolved, and all critical artifacts—including the generated adversarial concept vectors and the raw CSV audit logs—were saved locally to the file system. This ensures the data is backed up and accessible for future reporting without needing to recompute the entire attack pipeline.

In [7]:
# ==========================================
# STEP 8: Save Vectors and Audit Log Locally
# ==========================================
import os
from pathlib import Path
import torch

print("💾 Saving Task 4 artifacts...")

# Find the root project directory safely
root_dir = Path.cwd()
while not (root_dir / 'app.py').exists() and root_dir != root_dir.parent:
    root_dir = root_dir.parent

# Point directly to the task4_ folder 
# (Based on your folder structure, assuming it sits next to task5_)
save_dir = root_dir / 'ml_extensions' / 'task4_'
save_dir.mkdir(parents=True, exist_ok=True)

# 1. Save the Vectors (Centroids + Adversarial)
vector_path = save_dir / 'task4_steering_vectors.pt'
torch.save({
    'centroids': concept_vectors,
    'adversarial_vectors': all_adversarial_vectors
}, vector_path)
print(f"   ✅ Saved Vectors: {vector_path}")

# 2. Save the Audit Results as a CSV
csv_path = save_dir / 'task4_steering_audit.csv'
df_audit.to_csv(csv_path, index=False)
print(f"   ✅ Saved Audit Data: {csv_path}")

print("\n🎉 Task 4 artifacts successfully saved to the task4_ folder!")

💾 Saving Task 4 artifacts...
   ✅ Saved Vectors: /Users/emaheshwari/Project2/ml_extensions/task4_/task4_steering_vectors.pt
   ✅ Saved Audit Data: /Users/emaheshwari/Project2/ml_extensions/task4_/task4_steering_audit.csv

🎉 Task 4 artifacts successfully saved to the task4_ folder!
